# SMA-RSI Trend Following Backtest

Bu not defteri Yahoo Finance verisiyle sinyal ve maliyet sonrasi sermaye egrisini uretir. Sonuclar egitim amaclidir ve yatirim tavsiyesi degildir.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from backtest.engine import run_long_only_backtest
from data.loader import download_ohlcv
from strategies.trend_following import SmaRsiConfig, sma_rsi_signals

In [ ]:
ticker = 'AAPL'
prices = download_ohlcv(ticker, start='2021-01-01', end='2024-01-01')
prices.tail()

In [ ]:
config = SmaRsiConfig(short_window=20, long_window=50, rsi_window=14)
signals = sma_rsi_signals(prices['close'], config)
result = run_long_only_backtest(
    prices['close'],
    signals['position'],
    transaction_cost=0.0005,
)
pd.Series(result.metrics, name=ticker).to_frame()

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(prices.index, prices['close'], label=f'{ticker} close', color='black')
axes[0].plot(signals.index, signals['sma_20'], label='SMA 20')
axes[0].plot(signals.index, signals['sma_50'], label='SMA 50')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[1].plot(result.equity_curve.index, result.equity_curve, label='Strategy equity', color='tab:green')
axes[1].set_ylabel('Equity')
axes[1].set_xlabel('Date')
axes[1].legend()
figure.tight_layout()